In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Định nghĩa kiến trúc Baseline CNN
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        # Ảnh đầu vào: 1 kênh (Grayscale), kích thước 128x128
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Sau 2 lần MaxPool (chia đôi kích thước), ảnh 128x128 trở thành 32x32
        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 2) # 2 đầu ra: def_front và ok_front
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 32 * 32) # Làm phẳng (flatten) tensor
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Khởi tạo mô hình
model = BaselineCNN()

# Thiết lập hàm mất mát (Loss Function) và bộ tối ưu hóa (Optimizer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


BaselineCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path

# 1. Thiết lập lại DataLoader trong notebook mới này
dataset_root = Path.cwd().parent / "data" / "casting_data" / "casting_data"
if Path.cwd().name != "notebooks":
    dataset_root = Path.cwd() / "data" / "casting_data" / "casting_data"

data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = datasets.ImageFolder(root=dataset_root / 'train', transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. Vòng lặp huấn luyện (Training Loop)
epochs = 3
print("🚀 Bắt đầu huấn luyện mô hình...")

for epoch in range(epochs):
    model.train() # Đặt mô hình ở chế độ huấn luyện
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        optimizer.zero_grad() # Xóa gradient cũ
        
        outputs = model(images) # Lan truyền tiến
        loss = criterion(outputs, labels) # Tính Loss
        
        loss.backward() # Lan truyền ngược
        optimizer.step() # Cập nhật trọng số
        
        running_loss += loss.item()
        
        # Tính độ chính xác tạm thời
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("✅ Huấn luyện hoàn tất!")


🚀 Bắt đầu huấn luyện mô hình...
Epoch 1/3 - Loss: 0.2682 | Accuracy: 88.30%
Epoch 2/3 - Loss: 0.0799 | Accuracy: 97.30%
Epoch 3/3 - Loss: 0.0413 | Accuracy: 98.82%
✅ Huấn luyện hoàn tất!
